# Itemwise Analysis: Cross-ISI Consistency & Power

Compares itemwise hit/false-alarm rates between short and long ISI conditions.

**Analyses:**
1. Load single-ISI experiments (short and long)
2. Split-half reliability within each condition
3. Cross-ISI itemwise correlations (hits and false alarms)
4. Noise ceiling estimation (Spearman-Brown corrected)
5. Power analysis: how correlation improves with sample size

In [ ]:
import sys, os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..', '..')))

import numpy as np
import matplotlib.pyplot as plt

from utls.loading import load_results_with_exclusion_no_dropping, move_sequences_to_used
from utls.dprime import recompute_dprime_by_isi_per_subject
from utls.reliability import compute_itemwise_split_half_reliability

from utls.data_loading import load_single_isi, HR_TASK_NAMES, make_save_dir
from utls.human_analysis import (
    run_analysis, yes_rate_from_exps,
    cross_experiment_itemwise_correlation,
    spearman_brown, noise_ceiling,
    itemwise_power_analysis,
)
from utls.human_plotting import plot_itemwise_scatter, plot_power_curve

## Configuration

In [ ]:
# ── CONFIGURATION ──────────────────────────────────────────────
TASK = "atexts-len120"     # Options: "ind-nature-len120", "global-music-len120", "atexts-len120", "nhs-region-len120"
ISI_SHORT = 2
ISI_LONG = 16

FIGURES_BASE = "/om2/user/bjmedina/auditory-memory/memory/figures/human-results"
TASK_HR_NAME = HR_TASK_NAMES[TASK]

save_dir = make_save_dir(FIGURES_BASE, TASK, sub=f"cross-isi-{ISI_SHORT}-vs-{ISI_LONG}")
print(f"Task: {TASK_HR_NAME}")
print(f"Short ISI: {ISI_SHORT}, Long ISI: {ISI_LONG}")
print(f"Figures: {save_dir}")

## 1. Load Short-ISI and Long-ISI Experiments

In [ ]:
data_short = load_single_isi(
    task=TASK, isi=ISI_SHORT,
    load_fn=load_results_with_exclusion_no_dropping,
)
print(f"Short ISI ({ISI_SHORT}): N = {data_short['N']}")

data_long = load_single_isi(
    task=TASK, isi=ISI_LONG,
    load_fn=load_results_with_exclusion_no_dropping,
)
print(f"Long ISI ({ISI_LONG}):  N = {data_long['N']}")

## 2. Within-Condition Split-Half Reliability

In [ ]:
results_short = compute_itemwise_split_half_reliability(
    data_short['exps'], min_isi=ISI_SHORT, max_isi=ISI_SHORT
)
results_long = compute_itemwise_split_half_reliability(
    data_long['exps'], min_isi=ISI_LONG, max_isi=ISI_LONG
)

for label, res, isi in [("Short", results_short, ISI_SHORT), ("Long", results_long, ISI_LONG)]:
    r_h, sem_h = res['split_half_reliability']['hits']
    r_f, sem_f = res['split_half_reliability']['false_alarms']
    print(f"{label} ISI={isi}:  r_hits={r_h:.3f} (sem={sem_h:.3f}),  r_fa={r_f:.3f} (sem={sem_f:.3f})")

## 3. Cross-ISI Itemwise Correlations

In [ ]:
corr_hits = cross_experiment_itemwise_correlation(results_short, results_long, 'hits')
corr_fa = cross_experiment_itemwise_correlation(results_short, results_long, 'false_alarms')

print(f"Itemwise hit-rate correlation    (ISI {ISI_SHORT} vs {ISI_LONG}): r = {corr_hits['r']:.3f}  (N_items = {corr_hits['n_items']})")
print(f"Itemwise false-alarm correlation (ISI {ISI_SHORT} vs {ISI_LONG}): r = {corr_fa['r']:.3f}  (N_items = {corr_fa['n_items']})")

## 4. Noise Ceiling

In [ ]:
r_hit_short = results_short['split_half_reliability']['hits'][0]
r_hit_long  = results_long['split_half_reliability']['hits'][0]
r_fa_short  = results_short['split_half_reliability']['false_alarms'][0]
r_fa_long   = results_long['split_half_reliability']['false_alarms'][0]

nc_hits = noise_ceiling(r_hit_short, r_hit_long)
nc_fa   = noise_ceiling(r_fa_short, r_fa_long)

print(f"Spearman-Brown corrected reliabilities:")
print(f"  Hits:  short={spearman_brown(r_hit_short):.3f}, long={spearman_brown(r_hit_long):.3f}")
print(f"  FA:    short={spearman_brown(r_fa_short):.3f}, long={spearman_brown(r_fa_long):.3f}")
print(f"")
print(f"Noise ceiling (hits):           {nc_hits:.3f}")
print(f"Noise ceiling (false alarms):   {nc_fa:.3f}")

## 5. Itemwise Scatter Plots

In [ ]:
plot_itemwise_scatter(
    corr_hits['rates_a'], corr_hits['rates_b'], corr_hits['r'],
    kind='hits',
    xlabel=f'ISI {ISI_SHORT}', ylabel=f'ISI {ISI_LONG}',
    noise_ceiling=nc_hits,
    title=f"{TASK_HR_NAME}: Itemwise Hit Rates (ISI {ISI_SHORT} vs {ISI_LONG})",
    save_path=os.path.join(save_dir, f"itemwise-hits_isi{ISI_SHORT}_vs_isi{ISI_LONG}.png"),
)

In [ ]:
plot_itemwise_scatter(
    corr_fa['rates_a'], corr_fa['rates_b'], corr_fa['r'],
    kind='false_alarms',
    xlabel=f'ISI {ISI_SHORT}', ylabel=f'ISI {ISI_LONG}',
    noise_ceiling=nc_fa,
    title=f"{TASK_HR_NAME}: Itemwise False-Alarm Rates (ISI {ISI_SHORT} vs {ISI_LONG})",
    save_path=os.path.join(save_dir, f"itemwise-fa_isi{ISI_SHORT}_vs_isi{ISI_LONG}.png"),
)

## 6. Power Analysis: Sample Size vs Cross-ISI Correlation

In [ ]:
pwr_hits = itemwise_power_analysis(
    results_short, results_long, kind='hits', n_boot=300, step=5
)

plot_power_curve(
    pwr_hits, kind='hits',
    title=f"{TASK_HR_NAME}: Power Curve — Itemwise Hits (ISI {ISI_SHORT} vs {ISI_LONG})",
    save_path=os.path.join(save_dir, f"power-curve-hits_isi{ISI_SHORT}_vs_{ISI_LONG}.png"),
)

In [ ]:
pwr_fa = itemwise_power_analysis(
    results_short, results_long, kind='false_alarms', n_boot=300, step=5
)

plot_power_curve(
    pwr_fa, kind='false_alarms',
    title=f"{TASK_HR_NAME}: Power Curve — Itemwise FA (ISI {ISI_SHORT} vs {ISI_LONG})",
    save_path=os.path.join(save_dir, f"power-curve-fa_isi{ISI_SHORT}_vs_{ISI_LONG}.png"),
)

## Summary

In [ ]:
print(f"=== {TASK_HR_NAME}: Cross-ISI Itemwise Summary ===")
print(f"Short ISI={ISI_SHORT} (N={data_short['N']}), Long ISI={ISI_LONG} (N={data_long['N']})")
print(f"")
print(f"Within-condition split-half reliability:")
print(f"  Short hits: r={r_hit_short:.3f}  |  Long hits: r={r_hit_long:.3f}")
print(f"  Short FA:   r={r_fa_short:.3f}  |  Long FA:   r={r_fa_long:.3f}")
print(f"")
print(f"Cross-ISI correlations:")
print(f"  Hits: r={corr_hits['r']:.3f}  (noise ceiling: {nc_hits:.3f})")
print(f"  FA:   r={corr_fa['r']:.3f}  (noise ceiling: {nc_fa:.3f})")